# SOXX 2019 NPORT-P XML 原件样本

唯一固定原始 XML 来自已核对的 SOXX series 2019-09-30 申报索引；只检查基金身份与投资条目结构，不计算持仓权重或收益。


In [1]:
import json,hashlib,subprocess,tarfile,tempfile,shlex,xml.etree.ElementTree as ET,collections
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_soxx_2019_nport_source_specimen_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists()
assert sha(root/plan['parent_index']['path'])==plan['parent_index']['sha256']
rawdir=root/'data/snapshots/stage1_soxx_2019_nport_source_specimen_20261023';rawdir.mkdir(parents=True,exist_ok=True);assert not any(rawdir.iterdir())
raw=rawdir/'primary_doc.xml';headers=rawdir/'primary_doc.headers';started=datetime.now(timezone.utc).isoformat()
(card/'attempt_started.json').write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'started_at_utc':started,'GET_attempts_started':1,'model_fits':0,'native_accounts':0},indent=2)+'\n')
cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(plan['transport']['connect_timeout_seconds']),'--max-time',str(plan['transport']['total_timeout_seconds']),'--max-filesize',str(plan['transport']['max_response_bytes']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(raw),'-A',plan['transport']['user_agent'],'-H','Accept: application/xml','-w','%{http_code} %{url_effective}',plan['url']]
try:
 run=subprocess.run(cmd,capture_output=True,text=True,timeout=60);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
 receipt={'url':plan['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
except Exception as exc:receipt={'url':plan['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
receipt.update(raw_sha256=sha(raw) if raw.exists() else None,raw_bytes=raw.stat().st_size if raw.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
(card/'capture_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-2019-nport-specimen-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for p in [raw,headers,card/'capture_receipt.json',card/'attempt_started.json']:
   if p.exists():out.add(p,arcname=p.name)
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='soxx-nport-specimen-readback-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:
   run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for p in [raw,headers]:
    if p.exists():
     member=source.extractfile(p.name);assert member and hashlib.sha256(member.read()).hexdigest()==sha(p)
structure=None;parse_error=None
if receipt.get('http_status')==200 and receipt.get('curl_exit_code')==0 and raw.exists():
 try:
  root_xml=ET.fromstring(raw.read_bytes());local=lambda t:t.rsplit('}',1)[-1]
  tags=collections.Counter(local(e.tag) for e in root_xml.iter())
  identities={k:sorted({(e.text or '').strip() for e in root_xml.iter() if local(e.tag).lower()==k and (e.text or '').strip()}) for k in ['seriesid','classid','cik','repPd','reportdate','seriesname','registrantname']}
  investment_tags={k:v for k,v in tags.items() if any(word in k.lower() for word in ['invstorsec','holding','investment'])}
  structure={'root_tag':local(root_xml.tag),'element_count':sum(tags.values()),'identity_fields':identities,'investment_related_tag_counts':investment_tags,'top_level_tags':[local(e.tag) for e in list(root_xml)[:20]]}
 except Exception as exc:parse_error=type(exc).__name__+':'+str(exc)[:250]
status='one_original_NPORT_XML_structure_qualified_source_only' if structure and not parse_error and any('invst' in k.lower() or 'holding' in k.lower() for k in structure['investment_related_tag_counts']) and 'S000004354' in sum(structure['identity_fields'].values(),[]) else 'source_or_schema_unqualified'
result={'status':status,'plan_sha256':sha(card/'plan.json'),'transport':receipt,'structure':structure,'parse_error':parse_error,'nas_archive_sha256':digest,'nas_archive_bytes':evidence['size_bytes'],'nas_remote_and_scoped_restore_verified':True,'all_2019_2022_filings_qualified':False,'holding_values_parsed':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','parse_error','nas_archive_sha256')})


{'status': 'one_original_NPORT_XML_structure_qualified_source_only', 'parse_error': None, 'nas_archive_sha256': '4a5fc73692fcf3b3d43171a2dc156f98c2bb753c53d5ab0e07595ea193503308'}


## 中文结论

2019-09-30 原始 SOXX NPORT-P XML 38,831 字节、SHA-256 `951d162d4e996b74a7e1818e13c4c118c348320a95e54a72a4158a20c88d0e73`，NAS 归档和定点恢复通过。原 XML 有正确 CIK/series/class、34 个投资条目；初版漏核 `repPdDate`，零网络修正另见 schema_semantics_repair.ipynb。未计算持仓集中度、收益或模型。
